# Food Long-Tail OOD Challenge — Vanilla ResNet Baseline

Plainest possible baseline:
1. ResNet-50 ImageNet pretrained, fine-tune on the full training set (no long-tail tricks).
2. Predict the argmax over 101 known classes for every test image.
3. Write `submission.csv`.

Expected behavior: the model can never predict the `unknown` class (101), so the ~25% OOD test samples will all be wrong — capping the achievable Top-1 accuracy at roughly 75%. This baseline establishes the floor; better submissions add long-tail and OOD handling on top.

In [4]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# 直接拉取你们的比赛数据
!kaggle competitions download -c ow-food

# 将下载好的压缩包解压到专门的文件夹中
!unzip -q ow-food.zip -d /content/Food-Classification

print("✅ 数据集极速下载与解压完毕！")

100% 853M/853M [00:54<00:00, 16.5MB/s]

✅ 数据集极速下载与解压完毕！


## 1. Setup

In [13]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from tqdm.auto import tqdm

# 数据根目录与提交输出路径
DATA_ROOT = Path('/content/Food-Classification')
OUT_SUB = DATA_ROOT / 'submission_1.csv'

# 已知类数：模型只学 0..100，OOD（101）由后处理判定
NUM_KNOWN = 101
IMG_SIZE = 224          # ResNet 默认输入；图像本身是 256，会被 resize 到 224
BATCH_SIZE = 64
NUM_WORKERS = 4
EPOCHS = 5              # baseline 跑通即可，不追求收敛
LR = 1e-3
SEED = 42

# 自动选设备：优先 CUDA → Apple MPS → CPU
if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
elif torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
else:
    DEVICE = torch.device('cpu')
print('Device:', DEVICE)

torch.manual_seed(SEED); np.random.seed(SEED)

Device: cuda


## 2. Dataset

In [6]:
IMNET_MEAN = [0.485, 0.456, 0.406]
IMNET_STD = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(IMNET_MEAN, IMNET_STD),
])
eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMNET_MEAN, IMNET_STD),
])

class FoodDataset(Dataset):
    """从 csv (image_id[, label]) + 图像目录加载样本。
    has_label=False 用于测试集，__getitem__ 第二项返回 image_id 以便回填。"""

    def __init__(self, df, img_dir, transform, has_label):
        self.df = df.reset_index(drop=True)
        self.img_dir = Path(img_dir)
        self.transform = transform
        self.has_label = has_label

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = Image.open(self.img_dir / f"{row['image_id']}.jpg").convert('RGB')
        img = self.transform(img)
        if self.has_label:
            return img, int(row['label'])
        return img, row['image_id']

# 现在程序知道 DATA_ROOT 是什么了，再安全地读取 CSV
train_df = pd.read_csv(DATA_ROOT / 'train.csv')
test_df = pd.read_csv(DATA_ROOT / 'test.csv')
print(f'✅ 表格读取成功！train: {len(train_df)}, test: {len(test_df)}')

# 加载图片数据集
tr_ds = FoodDataset(train_df, DATA_ROOT / 'train_images' / 'train_images', train_tf, has_label=True)
test_ds = FoodDataset(test_df, DATA_ROOT / 'test_images' / 'test_images', eval_tf, has_label=False)

# pin_memory 仅在 CUDA 下能加速，MPS/CPU 无效
tr_loader = DataLoader(tr_ds, batch_size=BATCH_SIZE, shuffle=True,
                       num_workers=NUM_WORKERS, pin_memory=(DEVICE.type=='cuda'))
# 推理时 batch 翻倍，无需反向传播显存压力小
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE*2, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=(DEVICE.type=='cuda'))
print('✅ 数据流水线组装完毕，可以开始训练了！')

✅ 表格读取成功！train: 32856, test: 20179
✅ 数据流水线组装完毕，可以开始训练了！


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


## 3. Model

In [7]:
# ResNet-50 + ImageNet V2 权重（精度比 V1 高约 1%）
# 替换 fc 层，输出维度改为 101（已知类数）
model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
model.fc = nn.Linear(model.fc.in_features, NUM_KNOWN)
model = model.to(DEVICE)
print(f'ResNet-50, output dim {NUM_KNOWN}')

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 144MB/s]


ResNet-50, output dim 101


## 4. Train

In [8]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

for epoch in range(1, EPOCHS + 1):
    model.train()
    t0 = time.time()
    total, correct, loss_sum = 0, 0, 0.0
    bar = tqdm(tr_loader, desc=f'epoch {epoch}/{EPOCHS}', leave=False)
    for x, y in bar:
        x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
        logits = model(x)
        loss = criterion(logits, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        loss_sum += loss.item() * x.size(0)
        correct += (logits.argmax(1) == y).sum().item()
        total += x.size(0)
        bar.set_postfix(loss=loss_sum/total, acc=correct/total)
    scheduler.step()
    print(f'Epoch {epoch}/{EPOCHS} | loss {loss_sum/total:.3f} | train acc {correct/total:.3f} | {time.time()-t0:.1f}s')

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


epoch 1/5:   0%|          | 0/514 [00:00<?, ?it/s]

Epoch 1/5 | loss 2.131 | train acc 0.494 | 321.4s


epoch 2/5:   0%|          | 0/514 [00:00<?, ?it/s]

Epoch 2/5 | loss 1.376 | train acc 0.654 | 338.5s


...

Epoch 3/5 | loss 0.932 | train acc 0.753 | 355.7s


epoch 4/5:   0%|          | 0/514 [00:00<?, ?it/s]

Epoch 4/5 | loss 0.520 | train acc 0.862 | 339.3s


epoch 5/5:   0%|          | 0/514 [00:00<?, ?it/s]

Epoch 5/5 | loss 0.234 | train acc 0.940 | 338.8s


## 5. Predict on test set

In [12]:
# MSP 分位数阈值法：
# 先计算测试集中每张图片的最大 softmax 概率，
# 再将最大概率最低的 25% 样本判定为 unknown，也就是标签 101。

OOD_LABEL = 101
OOD_RATIO = 0.25

model.eval()

all_ids = []
all_preds = []
all_msps = []

with torch.no_grad():
    for x, ids in tqdm(test_loader, desc='predict'):
        x = x.to(DEVICE, non_blocking=True)

        probs = model(x).softmax(dim=1)
        msp, pred = probs.max(dim=1)

        all_ids.extend(ids)
        all_preds.extend(pred.cpu().numpy().tolist())
        all_msps.extend(msp.cpu().numpy().tolist())

# 转成 numpy，方便按分位数处理
all_preds = np.array(all_preds)
all_msps = np.array(all_msps)

# 取 MSP 最低 25% 的分界点
MSP_THRESHOLD = np.quantile(all_msps, OOD_RATIO)
print("MSP 25% threshold:", MSP_THRESHOLD)

# 最大 softmax 概率低于等于该阈值的样本，判为 unknown
final_preds = all_preds.copy()
final_preds[all_msps <= MSP_THRESHOLD] = OOD_LABEL

# 为了和后面的 submission 代码保持一致，仍然叫 all_preds
all_preds = final_preds.tolist()

print(f'Predicted {len(all_ids)} samples')
print(f'Predicted unknown samples: {(np.array(all_preds) == OOD_LABEL).sum()}')
print(f'Unknown ratio: {(np.array(all_preds) == OOD_LABEL).mean():.4f}')

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


predict:   0%|          | 0/158 [00:00<?, ?it/s]

MSP 25% threshold: 0.4236603081226349
Predicted 20179 samples
Predicted unknown samples: 5045
Unknown ratio: 0.2500


## 6. Submission

In [14]:
sub = pd.DataFrame({'image_id': all_ids, 'label': all_preds})
# DataLoader 是 shuffle=False，但 num_workers>0 时返回顺序不保证完全一致
# 这里按 sample_submission.csv 的 image_id 顺序重排，确保提交对齐
order = pd.read_csv(DATA_ROOT / 'test.csv')['image_id'].tolist()
sub = sub.set_index('image_id').loc[order].reset_index()
sub.to_csv(OUT_SUB, index=False)
print(f'Wrote {OUT_SUB} ({len(sub)} rows)')
sub.head()

Wrote /content/Food-Classification/submission_1.csv (20179 rows)


,image_id,label
0,test_00000,89
1,test_00001,101
2,test_00002,101
3,test_00003,5
4,test_00004,63
